# Chain-of-Thought — Chain-of-Thought Prompting Elicits Reasoning in Large Language Models (2022)

_Papers · Transformers & LLMs_

**Show the model a few worked, step-by-step examples and it learns to reason out loud — but only once it is big enough.**

---

This notebook is a practice scaffold for the **Chain-of-Thought — Chain-of-Thought Prompting Elicits Reasoning in Large Language Models (2022)** lesson. We build a small error-compounding model that shows *why* writing out the steps can help, one piece at a time — no language model is run. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Model whole-problem success as a product of steps

This is an **illustration, not the paper's numbers**, and no language model is involved. The idea: a problem takes `k` correct steps, and the final answer is right only if *every* step is right. If each step succeeds independently with probability `p`, the whole-problem success is `p**k` — errors compound across steps. We define that function and use it on a 4-step problem.

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# OUR ILLUSTRATION -- NOT the paper's numbers, and NO language model is run.
# Toy model: a problem needs k correct steps; the whole answer is right only
# if EVERY step is right. One step is correct with probability p, so the
# whole-problem success is p ** k (errors compound across steps).
# ---------------------------------------------------------------------------
def whole_problem_success(p, k):
    return p ** k


k = 4                                  # a 4-step problem

### Step 2 — Recompute the worked example

Compare two regimes on that same 4-step problem. With the steps *hidden* the model gets each step right with probability 0.80; *writing each step out* lifts per-step success to 0.92. Watch how a modest per-step gain (0.80 → 0.92) compounds into a large whole-problem gain (0.41 → 0.72) — that compounding is the intuition behind chain-of-thought.

In [ ]:
# (1) WORKED EXAMPLE from the lesson, recomputed.
p_oneshot = 0.80                       # per-step success, steps hidden
p_written = 0.92                       # writing each step lifts per-step success
print("one-shot   (p=0.80, k=4): %.3f" % whole_problem_success(p_oneshot, k))
print("step-by-step(p=0.92, k=4): %.3f" % whole_problem_success(p_written, k))
# one-shot   (p=0.80, k=4): 0.410
# step-by-step(p=0.92, k=4): 0.716
# A modest per-step gain (0.80 -> 0.92) compounds into 0.41 -> 0.72 on the whole
# problem. (Made-up numbers; illustrates the FORM of the effect, not the paper.)

### Step 3 — Sweep per-step accuracy to see where it helps most

Now vary the per-step success `p` across a range and model "writing the steps" as a fixed per-step lift (capped at 1.0). Both columns use the same `p**k` form. The gap between one-shot and step-by-step is widest in the *middle* of the range — that is where decomposing the problem and lifting per-step accuracy buys the most.

In [ ]:
# (2) SWEEP per-step success p. The step-by-step curve assumes writing the steps
#     out adds a fixed per-step lift (capped at 1.0). Both use the SAME p**k form.
lift = 0.12                            # made-up per-step boost from writing steps
ps = np.linspace(0.50, 0.99, 6)        # six per-step success values
print("\n  p_step | one-shot p^k | step-by-step (p+lift)^k")
for p in ps:
    p2 = min(p + lift, 1.0)            # lifted per-step success, capped at 1.0
    base = whole_problem_success(p, k)
    lifted = whole_problem_success(p2, k)
    print("   %.2f  |    %.3f     |    %.3f" % (p, base, lifted))
#   p_step | one-shot p^k | step-by-step (p+lift)^k
#    0.50  |    0.062     |    0.148
#    0.60  |    0.130     |    0.269
#    0.70  |    0.240     |    0.452
#    0.80  |    0.410     |    0.716
#    0.89  |    0.627     |    1.000
#    0.99  |    0.961     |    1.000
# The gap is widest in the middle: that is where lifting per-step accuracy and
# decomposing the problem buys the most. ILLUSTRATION ONLY -- no model, not GSM8K.

## Visualize the data & results

_OUR ILLUSTRATION (no language model): if a problem needs k correct steps each solved with probability p, the whole-problem success is p**k. How does writing the steps out -- which we model as a fixed per-step lift -- change the whole-problem success as p varies? This visualizes WHY decomposition can help; it is NOT the paper's GSM8K number._

### Step 1 — Recompute the curve as arrays

Same illustration as above (no language model, not the paper's numbers), now vectorised so we can plot it. For a fixed array of per-step accuracies `p` we compute the one-shot curve `p**k` and the step-by-step curve `(p+lift)**k`, clipping the lifted accuracy at 1.0.

In [ ]:
import numpy as np

# OUR ILLUSTRATION -- no language model, NOT the paper's numbers.
# whole-problem success = p**k  (all k steps must be right; one step right w.p. p)
k = 4
lift = 0.12                                  # made-up per-step boost from writing
p = np.array([0.50, 0.60, 0.70, 0.80, 0.89, 0.99])

oneshot = p ** k
p_lifted = np.minimum(p + lift, 1.0)         # lifted per-step success, capped at 1.0
stepbystep = p_lifted ** k

### Step 2 — Print the table and the worked example

Lay the two curves side by side and re-show the original worked example (`0.8**4` vs `0.92**4`). The step-by-step column dominates everywhere and the gap is widest mid-range — the same decomposition intuition, just tabulated. This is the form of the effect, not the paper's measured GSM8K accuracy.

In [ ]:
print(" p_step | one-shot p^k | step-by-step (p+lift)^k")
for pi, a, b in zip(p, oneshot, stepbystep):
    print("  %.2f  |    %.4f    |    %.4f" % (pi, a, b))
print("\nworked example: 0.8^4 = %.3f,  0.92^4 = %.3f" % (0.8 ** 4, 0.92 ** 4))
#  p_step | one-shot p^k | step-by-step (p+lift)^k
#   0.50  |    0.0625    |    0.1478
#   0.60  |    0.1296    |    0.2687
#   0.70  |    0.2401    |    0.4521
#   0.80  |    0.4096    |    0.7164
#   0.89  |    0.6274    |    1.0000
#   0.99  |    0.9606    |    1.0000
#
# worked example: 0.8^4 = 0.410,  0.92^4 = 0.716
# The step-by-step curve dominates; the gap is widest mid-range. ILLUSTRATION
# ONLY -- this is the decomposition INTUITION, not the paper's measured accuracy.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compounding steps. In the toy error model, a problem needs $k$ correct steps, each right with
            probability $p$, and the answer is right only if all steps are right. (a) Whole-problem success for
            $p = 0.9$, $k = 5$? (b) For the same per-step $p = 0.9$, does a 3-step problem or an 8-step problem have a
            higher whole-problem success, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $P = p^k$. For (a): $0.9^5$. — _The answer is correct only if every one of the $k$ independent steps is correct, so the probabilities multiply._
- Compute $0.9^5$: $0.9^2 = 0.81$, $0.9^4 = 0.81^2 = 0.6561$, times $0.9$ gives $\approx 0.590$. — _Square, square again, then one more factor &mdash; that is five factors of $0.9$._
- Compare exponents: $0.9^3 = 0.729$ vs $0.9^8 \approx 0.430$. Fewer steps wins. — _Each extra step multiplies by another $p < 1$, so more steps always lowers the product._

**Answer:** (a) $0.9^5 \approx 0.590$, about 59%. (b) The 3-step problem
                 ($0.9^3 = 0.729$, about $73\%$) beats the 8-step one ($0.9^8 \approx 0.430$, about $43\%$):
                 every additional step multiplies by another factor below 1, so success decays with length. This is
                 why raising per-step accuracy (clearer, checkable steps) matters so much for long problems.
                 (Toy illustration, not a paper number.)

</details>

**Problem 2.** The emergence claim. A colleague tries chain-of-thought prompting on a small (1B-parameter)
            model and sees worse accuracy than plain prompting. Is this consistent with the paper? Quote the
            relevant finding and explain what the colleague should change.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall &sect;3.2: chain of thought is "an emergent ability of model scale," helping only at "$\sim$100B parameters." — _The paper explicitly states the benefit is scale-gated, not universal._
- Recall the small-model failure mode: they "produced fluent but illogical chains of thought, leading to lower performance than standard prompting." — _This predicts exactly the colleague's observation &mdash; worse than plain prompting on a small model._
- Conclude: try a much larger model (around or above 100B parameters); the method is not expected to help at 1B. — _The capability is emergent, so it appears only past the scale threshold._

**Answer:** Yes, fully consistent. The paper states chain of thought "is an emergent ability of model scale"
                 and "does not positively impact performance for small models, and only yields performance gains when
                 used with models of $\sim$100B parameters" (&sect;3.2). Small models "produced fluent but illogical
                 chains of thought, leading to lower performance than standard prompting" &mdash; exactly what the
                 colleague saw. The fix is scale: use a model around or above 100B (billion) parameters. At 1B,
                 the result is expected.

</details>

**Problem 3.** Per-step gain vs. more steps (ablation-style). Two strategies for a problem: (A) keep it as a
            single hidden computation with success $p = 0.65$; (B) decompose it into $k = 3$ written steps, which
            raises per-step success to $p' = 0.85$ but now all three must be right. Which strategy wins, and what does
            this say about why chain of thought helps?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Strategy A success: a single step, so $P_A = 0.65$. — _One hidden computation, one chance, no decomposition._
- Strategy B success: $P_B = p'^k = 0.85^3$. Compute $0.85^2 = 0.7225$, times $0.85 \approx 0.614$. — _Three independent written steps, all must be correct, so multiply three factors of $0.85$._
- Compare: $0.65$ vs $0.614$. Here A barely wins, because the per-step lift ($0.65 \to 0.85$) was not big enough to overcome three-way compounding. — _Decomposition only pays off when the per-step accuracy gain outweighs the cost of needing every step right._

**Answer:** $P_A = 0.65$; $P_B = 0.85^3 \approx 0.614$. Strategy A narrowly wins, so a $0.65 \to 0.85$
                 per-step lift is not enough for three steps. The lesson: chain of thought helps when writing
                 the steps raises per-step accuracy enough to beat the compounding penalty of needing all
                 steps right. If decomposition makes each step much more reliable, the product wins; if the lift is
                 marginal, splitting a problem into more steps can even hurt. (Toy illustration, not the paper's
                 measurement &mdash; the paper's actual gate on "enough" is model scale, &sect;3.2.)

</details>